# Machine Learning — Lab 12
## Principal Component Analysis and Unsupervised Data Analysis

**Main Course Learning Outcomes — CLO2, CLO4, CLO5**

- **CLO2:** Analyze datasets and apply appropriate preprocessing, transformation, and feature engineering techniques.
- **CLO4:** Apply supervised and unsupervised learning techniques to solve practical problems and interpret their outcomes.
- **CLO5:** Evaluate machine learning models using appropriate metrics and justify decisions based on performance trade-offs and real-world context.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

> **Assessment principle:** Reducing dimensions is not enough. Full credit requires you to explain **what PCA preserves, why scaling matters, how principal components are constructed, how much variance is retained, and whether the reduced representation is useful for visualization or clustering**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. PCA mechanics on a tiny dataset | 25 min | Center data, compute covariance, eigenvalues, eigenvectors, and projection |
| 2. Practical high-dimensional dataset | 15 min | Inspect correlated numerical features and scaling |
| 3. Fit PCA | 20 min | Compute components, scores, and explained variance |
| 4. Choose the number of components | 20 min | Use cumulative explained variance and reconstruction reasoning |
| 5. Interpret loadings | 15 min | Explain what selected components represent |
| 6. PCA + clustering | 15 min | Compare K-Means before and after dimensionality reduction |
| 7. Challenge, debugging & viva | 10 min | Diagnose common PCA workflow mistakes |
| **Total** | **120 min** | |

### Main idea

$$
\boxed{
\text{Correlated features}
\rightarrow
\text{center/scale}
\rightarrow
\text{principal directions}
\rightarrow
\text{projection}
\rightarrow
\text{lower-dimensional representation}
}
$$

## Learning Objectives

By the end of this lab, you should be able to:

1. explain why dimensionality reduction may be useful;
2. distinguish feature selection from feature extraction;
3. center and standardize numerical data;
4. compute a covariance matrix for a small dataset;
5. identify principal directions from eigenvalues and eigenvectors;
6. project observations onto principal components;
7. interpret explained variance and cumulative explained variance;
8. select a reasonable number of components;
9. interpret PCA loadings carefully;
10. evaluate whether PCA helps visualization and clustering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

print("Machine Learning Lab 12 environment ready.")

# Part I — Why Dimensionality Reduction?

Suppose a dataset contains:

$$
p=20
$$

features, many of which are strongly correlated.

Possible problems include:

- difficult visualization;
- redundant information;
- slower computation;
- unstable distance calculations;
- harder interpretation;
- increased risk of fitting noise.

PCA creates a new representation:

$$
X\in\mathbb{R}^{n\times p}
\rightarrow
Z\in\mathbb{R}^{n\times q},
\qquad q<p.
$$

The new features are called **principal components**.

## Task 1.1 — Feature Selection vs. Feature Extraction

Complete:

| Method | Keeps original features? | Creates new features? | Example |
|---|---|---|---|
| Feature selection |  |  |  |
| Feature extraction |  |  |  |

Then explain why PCA is **feature extraction**, not feature selection.

# Part II — PCA by Hand on a Tiny Dataset

We begin with a small 2D dataset.

The features are strongly correlated, so one principal direction should explain most of the variation.

In [ ]:
tiny = pd.DataFrame({
    "x1": [2.0, 3.0, 4.0, 5.0, 6.0],
    "x2": [2.2, 3.1, 4.2, 4.9, 6.1],
})

display(tiny)

plt.figure(figsize=(6, 5))
plt.scatter(tiny["x1"], tiny["x2"])
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Tiny Correlated Dataset")
plt.show()

## Task 2.1 — Predict Before Calculating

Before any PCA computation:

1. Are `x1` and `x2` positively or negatively correlated?
2. Do you expect the first principal component to run approximately from lower-left to upper-right?
3. Do you expect $PC_1$ or $PC_2$ to explain more variance?
4. Could one component summarize most of this dataset?

## Task 2.2 — Center the Data

PCA begins by centering each feature:

$$
x'_{ij}=x_{ij}-\bar{x}_j.
$$

Complete the function.

In [ ]:
def center_columns(X):
    X = np.asarray(X, dtype=float)

    # TODO: compute feature means and centered matrix.
    means = None
    centered = None

    return centered, means

In [ ]:
toy_centered, toy_means = center_columns(
    np.array([
        [1.0, 3.0],
        [3.0, 5.0],
    ])
)

assert np.allclose(toy_means, np.array([2.0, 4.0]))
assert np.allclose(
    toy_centered,
    np.array([
        [-1.0, -1.0],
        [1.0, 1.0],
    ])
)

print("Centering function tests passed.")

In [ ]:
X_tiny = tiny.to_numpy(dtype=float)

X_centered, means_tiny = center_columns(X_tiny)

print("Feature means:", np.round(means_tiny, 4))
print("Centered data:")
display(pd.DataFrame(
    X_centered,
    columns=["x1_centered", "x2_centered"]
))

## Task 2.3 — Check Centering

Answer:

1. What should the mean of each centered column be?
2. Why does PCA center the data?
3. What would happen geometrically if we did not remove the mean?

In [ ]:
print("Centered column means:", np.round(X_centered.mean(axis=0), 10))

## Task 2.4 — Covariance Matrix

For centered data, the sample covariance matrix is:

$$
\Sigma
=
\frac{1}{n-1}
X_c^\top X_c.
$$

Complete the function.

In [ ]:
def covariance_matrix(centered_X):
    centered_X = np.asarray(centered_X, dtype=float)
    n = centered_X.shape[0]

    # TODO: implement sample covariance matrix.
    cov = None

    return cov

In [ ]:
cov_toy = covariance_matrix(
    np.array([
        [-1.0, -1.0],
        [1.0, 1.0],
    ])
)

assert np.allclose(
    cov_toy,
    np.array([
        [2.0, 2.0],
        [2.0, 2.0],
    ])
)

print("Covariance function test passed.")

In [ ]:
cov_tiny = covariance_matrix(X_centered)

print("Covariance matrix:")
display(pd.DataFrame(
    cov_tiny,
    index=["x1", "x2"],
    columns=["x1", "x2"]
).round(4))

## Task 2.5 — Interpret Covariance

Answer:

1. Are the off-diagonal covariance values positive or negative?
2. What does their sign mean?
3. Why does strong covariance suggest redundancy between features?

# Part III — Eigenvalues and Principal Directions

PCA finds directions that capture variance.

For the covariance matrix:

$$
\Sigma v=\lambda v.
$$

- $v$ is an eigenvector: a principal direction;
- $\lambda$ is its eigenvalue: the variance captured along that direction.

The component with the largest eigenvalue becomes $PC_1$.

## Task 3.1 — Compute Eigenvalues and Eigenvectors

Use `np.linalg.eigh`, which is appropriate for symmetric covariance matrices.

Complete:

In [ ]:
def sorted_eigendecomposition(cov):
    cov = np.asarray(cov, dtype=float)

    # TODO: compute eigenvalues/eigenvectors.
    eigenvalues = None
    eigenvectors = None

    # TODO: sort from largest eigenvalue to smallest.
    order = None
    eigenvalues_sorted = None
    eigenvectors_sorted = None

    return eigenvalues_sorted, eigenvectors_sorted

In [ ]:
eigenvalues, eigenvectors = sorted_eigendecomposition(cov_tiny)

assert eigenvalues[0] >= eigenvalues[1]
assert eigenvectors.shape == (2, 2)

print("Eigen decomposition test passed.")
print("Eigenvalues:", np.round(eigenvalues, 6))
print("Eigenvectors:")
print(np.round(eigenvectors, 6))

## Task 3.2 — Interpret the Principal Directions

Answer:

1. Which eigenvalue is larger?
2. Which eigenvector corresponds to $PC_1$?
3. Does its direction match your visual prediction?
4. Why can the sign of an eigenvector be flipped without changing the PCA solution?

## Task 3.3 — Explained Variance Ratio

For component $j$:

$$
EVR_j
=
\frac{\lambda_j}{\sum_k\lambda_k}.
$$

Complete:

In [ ]:
def explained_variance_ratio(eigenvalues):
    eigenvalues = np.asarray(eigenvalues, dtype=float)

    # TODO: divide each eigenvalue by the total.
    ratios = None

    return ratios

In [ ]:
tiny_evr = explained_variance_ratio(eigenvalues)

assert abs(tiny_evr.sum() - 1.0) < 1e-12

print("Explained variance ratios:", np.round(tiny_evr, 6))

## Task 3.4 — Manual PCA Interpretation

Answer:

1. What percentage of variance is explained by $PC_1$?
2. How much remains for $PC_2$?
3. Would keeping only $PC_1$ be reasonable for a compact representation?
4. What information would be lost?

# Part IV — Project onto the First Principal Component

The PCA score of an observation on $PC_1$ is its projection:

$$
z_i=x_{c,i}^\top v_1.
$$

Complete:

In [ ]:
def project_onto_components(centered_X, components):
    centered_X = np.asarray(centered_X, dtype=float)
    components = np.asarray(components, dtype=float)

    # columns of components are principal directions.
    # TODO: project observations.
    projected = None

    return projected

In [ ]:
pc1_direction = eigenvectors[:, [0]]

tiny_pc1_scores = project_onto_components(
    X_centered,
    pc1_direction
)

assert tiny_pc1_scores.shape == (len(tiny), 1)

print("PC1 scores:")
print(np.round(tiny_pc1_scores.ravel(), 4))

## Task 4.1 — Interpret PCA Scores

Answer:

1. Which observations have negative $PC_1$ scores?
2. Which have positive scores?
3. What does the ordering of the scores tell you about movement along the main direction of variation?
4. Why are PCA scores new feature values rather than selected original features?

# Part V — Practical Dataset: Student Learning Profiles

We now use a higher-dimensional synthetic dataset describing student learning behavior.

Features:

- `attendance_pct`
- `study_hours_week`
- `lms_sessions`
- `forum_posts`
- `lab_completion_pct`
- `quiz_avg`
- `assignment_avg`
- `late_submissions`

Several variables are deliberately correlated because they are influenced by underlying factors such as **engagement** and **academic performance**.

There is no target label in this lab.

In [ ]:
rng = np.random.default_rng(3452)
n = 650

engagement = rng.normal(0, 1, n)
academic = 0.45 * engagement + rng.normal(0, 0.9, n)
time_pressure = rng.normal(0, 1, n)

attendance_pct = np.clip(
    80 + 9.5 * engagement - 3.0 * time_pressure + rng.normal(0, 5, n),
    35, 100
)

study_hours_week = np.clip(
    8 + 2.7 * engagement + 1.8 * academic + rng.normal(0, 2.2, n),
    0.5, 22
)

lms_sessions = np.clip(
    18 + 6.5 * engagement + rng.normal(0, 5, n),
    1, 55
)

forum_posts = np.clip(
    5 + 2.8 * engagement + rng.normal(0, 2.5, n),
    0, 20
)

lab_completion_pct = np.clip(
    78 + 9.0 * engagement + 4.5 * academic + rng.normal(0, 6, n),
    20, 100
)

quiz_avg = np.clip(
    70 + 5.0 * engagement + 10.0 * academic + rng.normal(0, 7, n),
    20, 100
)

assignment_avg = np.clip(
    72 + 4.0 * engagement + 9.0 * academic + rng.normal(0, 7, n),
    20, 100
)

late_submissions = np.clip(
    np.rint(3.0 - 1.2 * engagement + 1.1 * time_pressure + rng.normal(0, 1.0, n)),
    0, 9
).astype(int)

student_master = pd.DataFrame({
    "attendance_pct": np.round(attendance_pct, 1),
    "study_hours_week": np.round(study_hours_week, 1),
    "lms_sessions": np.round(lms_sessions, 1),
    "forum_posts": np.round(forum_posts, 1),
    "lab_completion_pct": np.round(lab_completion_pct, 1),
    "quiz_avg": np.round(quiz_avg, 1),
    "assignment_avg": np.round(assignment_avg, 1),
    "late_submissions": late_submissions,
})

print("Master dataset shape:", student_master.shape)
display(student_master.head())

## Task 5.1 — Personalized Working Dataset

Enter the last four digits of your student ID.

Your ID determines a reproducible sample of 520 students.

In [ ]:
# TODO: Replace None with the last four digits of your own student ID.
STUDENT_ID_LAST4 = None

if STUDENT_ID_LAST4 is None:
    raise ValueError("Enter the last four digits of your student ID.")

if not isinstance(STUDENT_ID_LAST4, int):
    raise TypeError("STUDENT_ID_LAST4 must be an integer.")

SEED = 12000 + (STUDENT_ID_LAST4 % 1000)

students = student_master.sample(
    n=520,
    random_state=SEED,
    replace=False
).reset_index(drop=True)

print("Your PCA seed:", SEED)
print("Working dataset shape:", students.shape)

## Task 5.2 — Inspect Correlation

Before running the correlation matrix, predict:

1. Which three variables are likely to be positively correlated?
2. Which variable may be negatively associated with engagement?
3. Why can correlated variables motivate PCA?

In [ ]:
corr = students.corr(numeric_only=True)

display(corr.round(3))

## Task 5.3 — Interpret the Correlation Matrix

Choose:

- one strong positive relationship;
- one moderate relationship;
- one negative relationship.

Explain what each suggests.

Then answer:

> Why does correlation among features not automatically mean one of them should simply be deleted?

# Part VI — Why Scaling Matters for PCA

PCA is variance-based.

Features with larger numerical variance can dominate the components.

We therefore standardize:

$$
z=\frac{x-\mu}{\sigma}.
$$

For this unsupervised exploratory analysis, we fit the scaler on the working dataset itself.

If PCA were part of a supervised predictive pipeline, the scaler and PCA would be fitted on the **training set only**.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(students)

scaled_df = pd.DataFrame(
    X_scaled,
    columns=students.columns
)

scale_report = pd.DataFrame({
    "mean": scaled_df.mean(),
    "std": scaled_df.std(ddof=0),
})

display(scale_report.round(4))

## Task 6.1 — Interpret Scaling

Answer:

1. Why are means approximately zero?
2. Why are standard deviations approximately one?
3. What would happen if `lms_sessions` had much larger numerical variance than all other features and we did not scale?
4. Does scaling make the features equally important in a conceptual sense?

# Part VII — Fit PCA

We fit PCA using all eight standardized features.

In [ ]:
pca_full = PCA()
Z_full = pca_full.fit_transform(X_scaled)

component_names = [
    f"PC{i+1}" for i in range(Z_full.shape[1])
]

explained = pd.DataFrame({
    "component": component_names,
    "explained_variance_ratio": pca_full.explained_variance_ratio_,
})

explained["cumulative_explained_variance"] = (
    explained["explained_variance_ratio"].cumsum()
)

display(explained.round(4))

## Task 7.1 — Interpret Explained Variance

Answer:

1. Which component explains the most variance?
2. How much variance is explained by the first two components together?
3. How many components are needed to preserve at least 80%?
4. How many are needed to preserve at least 90%?
5. Why is “95% explained variance” a guideline rather than a universal rule?

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(component_names) + 1),
    pca_full.explained_variance_ratio_,
    marker="o"
)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(component_names) + 1),
    np.cumsum(pca_full.explained_variance_ratio_),
    marker="o"
)
plt.axhline(0.80, linewidth=1)
plt.axhline(0.90, linewidth=1)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.show()

## Task 7.2 — Choose the Number of Components

Select a value of $q$ for a compact representation.

Your justification must mention:

- cumulative explained variance;
- dimensionality reduction achieved;
- interpretation needs;
- whether the purpose is visualization, clustering, or compression.

**Your selected $q$:**

# Part VIII — PCA Loadings

Each principal component is a linear combination of standardized original features:

$$
PC_j
=
a_{j1}x_1+a_{j2}x_2+\cdots+a_{jp}x_p.
$$

The coefficients are called **loadings**.

In [ ]:
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=students.columns,
    columns=component_names
)

display(loadings.round(4))

## Task 8.1 — Interpret $PC_1$

Inspect the largest absolute loadings in `PC1`.

Answer:

1. Which features contribute most strongly?
2. Are their signs mostly similar or mixed?
3. What broad latent idea might $PC_1$ represent?
4. Why is the name you assign only an interpretation?

In [ ]:
pc1_ranked = (
    loadings["PC1"]
    .abs()
    .sort_values(ascending=False)
)

print("Features ranked by absolute loading on PC1:")
display(pc1_ranked.to_frame("abs_loading"))

## Task 8.2 — Interpret $PC_2$

Repeat for `PC2`.

Explain:

- which variables dominate;
- what contrast the signs may represent;
- whether $PC_2$ appears different from $PC_1$.

Avoid saying that PCA “knows” the semantic meaning of the component.

In [ ]:
pc2_ranked = (
    loadings["PC2"]
    .abs()
    .sort_values(ascending=False)
)

display(pc2_ranked.to_frame("abs_loading"))

# Part IX — Visualize the Data in Two Principal Components

The first two PCA scores give a 2D representation:

$$
X\in\mathbb{R}^{520\times8}
\rightarrow
Z\in\mathbb{R}^{520\times2}.
$$

In [ ]:
Z2 = Z_full[:, :2]

pca2_df = pd.DataFrame(
    Z2,
    columns=["PC1", "PC2"]
)

plt.figure(figsize=(8, 5))
plt.scatter(
    pca2_df["PC1"],
    pca2_df["PC2"],
    alpha=0.7
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Student Profiles in the First Two Principal Components")
plt.show()

## Task 9.1 — Interpret the 2D Projection

Answer:

1. Do you see compact groups, gradients, or outliers?
2. How much total variance do PC1 and PC2 preserve?
3. What information is necessarily missing from this 2D view?
4. Why can two observations appear close in 2D but be farther apart in the full PCA space?

# Part X — Reconstruction Error

If we keep only $q<p$ principal components, some information is lost.

A reduced representation can be mapped approximately back to the original standardized feature space.

The reconstruction error helps quantify information loss.

## Task 10.1 — Compare Reconstruction for Different $q$

We will test:

$$
q\in\{1,2,3,4,5,6,7,8\}.
$$

For each $q$:

1. transform the standardized data;
2. inverse transform back to standardized feature space;
3. calculate mean squared reconstruction error.

In [ ]:
reconstruction_rows = []

for q in range(1, students.shape[1] + 1):
    pca_q = PCA(n_components=q)
    Z_q = pca_q.fit_transform(X_scaled)
    X_reconstructed = pca_q.inverse_transform(Z_q)

    mse = np.mean(
        (X_scaled - X_reconstructed) ** 2
    )

    reconstruction_rows.append({
        "components": q,
        "cumulative_variance": pca_q.explained_variance_ratio_.sum(),
        "reconstruction_mse": mse,
    })

reconstruction_table = pd.DataFrame(reconstruction_rows)
display(reconstruction_table.round(5))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    reconstruction_table["components"],
    reconstruction_table["reconstruction_mse"],
    marker="o"
)
plt.xlabel("Number of Components")
plt.ylabel("Reconstruction MSE")
plt.title("PCA Reconstruction Error")
plt.show()

## Task 10.2 — Interpret Reconstruction Error

Answer:

1. What happens to reconstruction error as more components are kept?
2. Why is reconstruction error exactly or nearly zero when all components are retained?
3. Is a lower reconstruction error always worth keeping more dimensions?
4. How is this a trade-off between compression and information preservation?

# Part XI — PCA Before K-Means

We now connect this lab to the previous clustering lab.

We compare K-Means using:

1. all 8 standardized features;
2. the first 2 principal components;
3. enough principal components to preserve at least 90% of variance.

We use the same:

$$
K=3
$$

for each representation so the comparison focuses on representation rather than cluster count.

In [ ]:
# Number of PCs required for at least 90% cumulative variance.
q90 = int(
    np.argmax(
        np.cumsum(
            pca_full.explained_variance_ratio_
        ) >= 0.90
    ) + 1
)

print("Components required for at least 90% variance:", q90)

In [ ]:
representations = {
    "All standardized features": X_scaled,
    "First 2 PCs": Z_full[:, :2],
    f"First {q90} PCs (>=90% variance)": Z_full[:, :q90],
}

cluster_rows = []
cluster_labels = {}

for name, representation in representations.items():
    model = KMeans(
        n_clusters=3,
        n_init=20,
        random_state=SEED
    )

    labels = model.fit_predict(representation)

    score = silhouette_score(
        representation,
        labels
    )

    cluster_rows.append({
        "representation": name,
        "dimensions": representation.shape[1],
        "silhouette": score,
        "inertia": model.inertia_,
    })

    cluster_labels[name] = labels

cluster_comparison = pd.DataFrame(cluster_rows)
display(cluster_comparison.round(4))

## Task 11.1 — Interpret PCA + K-Means

Answer:

1. Which representation has the highest silhouette score?
2. Does using only two PCs improve or worsen cluster separation?
3. Does retaining at least 90% variance preserve clustering quality well?
4. Why should inertia values not be compared directly across representations with different dimensionality and scaling?
5. Does PCA guarantee better clustering? Explain.

In [ ]:
labels_2d = cluster_labels["First 2 PCs"]

plt.figure(figsize=(8, 5))
plt.scatter(
    Z_full[:, 0],
    Z_full[:, 1],
    c=labels_2d,
    alpha=0.75
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters in the First Two Principal Components")
plt.show()

## Task 11.2 — Cluster Interpretation Warning

The colored groups are algorithmic clusters.

Answer:

1. Can we call them “high achievers”, “disengaged”, or “online learners” immediately?
2. What additional evidence would be needed?
3. How could PCA loadings help interpret the cluster positions?

# Part XII — Personalized PCA Challenge

Your student ID assigns an explained-variance target:

$$
75\%,\;80\%,\;85\%,\;90\%,\;95\%.
$$

You must choose the minimum number of components needed to reach your target.

In [ ]:
variance_targets = [0.75, 0.80, 0.85, 0.90, 0.95]
PERSONAL_TARGET = variance_targets[
    SEED % len(variance_targets)
]

cumulative = np.cumsum(
    pca_full.explained_variance_ratio_
)

PERSONAL_Q = int(
    np.argmax(cumulative >= PERSONAL_TARGET) + 1
)

print("Your target cumulative variance:", PERSONAL_TARGET)
print("Minimum components required:", PERSONAL_Q)
print("Actual cumulative variance retained:", round(cumulative[PERSONAL_Q - 1], 4))

## Task 12.1 — Predict Before Evaluating Your Representation

Before clustering your personalized PCA representation, predict:

1. how many original dimensions are removed;
2. whether the silhouette score will be close to the full-feature score;
3. whether interpretation becomes easier or harder;
4. whether your representation is suitable for 2D visualization.

In [ ]:
Z_personal = Z_full[:, :PERSONAL_Q]

personal_kmeans = KMeans(
    n_clusters=3,
    n_init=20,
    random_state=SEED
)

personal_labels = personal_kmeans.fit_predict(
    Z_personal
)

personal_silhouette = silhouette_score(
    Z_personal,
    personal_labels
)

print("Personal PCA dimensions:", PERSONAL_Q)
print("Retained variance:", round(cumulative[PERSONAL_Q - 1], 4))
print("Silhouette:", round(personal_silhouette, 4))

## Task 12.2 — Defend Your Representation

Write 4–6 sentences including:

- target explained variance;
- number of retained components;
- number of removed dimensions;
- actual retained variance;
- silhouette score;
- one benefit of the reduced representation;
- one limitation.

# Part XIII — Deliberate Debugging

Consider these four mistakes.

### Mistake A

PCA is applied to raw features with very different scales.

### Mistake B

An analyst keeps only the first two PCs because “two dimensions are easy to plot” without checking explained variance.

### Mistake C

A supervised model pipeline fits scaling and PCA on the full dataset before train/test splitting.

### Mistake D

The analyst says:

> “PC1 causes higher quiz scores.”

Explain why each statement or workflow is problematic.

## Task 13.1 — Fix a Projection Bug

Suppose:

- rows are observations;
- columns are centered features;
- columns of `components` are principal directions.

The correct projection is:

$$
Z=X_cV.
$$

Complete:

In [ ]:
def correct_projection(centered_X, components):
    centered_X = np.asarray(centered_X, dtype=float)
    components = np.asarray(components, dtype=float)

    # TODO: matrix multiply centered observations by principal directions.
    projected = None

    return projected

In [ ]:
toy_X = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
])

toy_V = np.eye(2)

assert np.allclose(
    correct_projection(toy_X, toy_V),
    toy_X
)

print("Projection debugging test passed.")

# Part XIV — Final Synthesis

PCA does not predict a target.

Its value depends on what the reduced representation is used for.

Possible goals include:

- visualization;
- compression;
- denoising;
- speeding later algorithms;
- reducing redundancy;
- improving some distance-based analyses.

But high explained variance does **not** guarantee high predictive value in a supervised task.

## Task 14.1 — Final Analysis Statement

Write a final summary that includes:

1. the first two PCs' cumulative explained variance;
2. your interpretation of $PC_1$ from loadings;
3. the minimum components required for at least 90% variance;
4. whether the 2D PCA representation was sufficient for clustering;
5. whether you would use PCA before K-Means for this dataset;
6. one reason PCA could remove information that matters to a supervised target.

Your answer should distinguish clearly between:

- preserving variance;
- preserving cluster structure;
- preserving predictive information.

# Individual Understanding Check

Your instructor may select one question for a 60–90 second explanation.

1. Why does PCA center the data?
2. Why is scaling often important before PCA?
3. What does an eigenvalue represent in PCA?
4. What does an eigenvector represent?
5. What is explained variance ratio?
6. Why are principal components new features rather than selected original features?
7. What do PCA loadings mean?
8. Why can a 2D PCA plot hide useful structure?
9. Why must PCA be fitted on training data only when used in a supervised pipeline?
10. For your personalized variance target, justify the number of components you kept.

You should be able to answer without reading a prepared paragraph.

# Reflection

Answer concisely in your own words.

1. What did the manual covariance/eigenvector calculation help you understand?
2. Why is PCA useful when features are correlated?
3. What is lost when dimensions are removed?
4. Why should explained variance not be confused with predictive importance?
5. What did the PCA + K-Means comparison teach you about representation?

**Your reflection:**

# Submission Checklist

Before submitting, confirm that your notebook contains:

- [ ] feature-selection vs. feature-extraction explanation;
- [ ] completed centering function;
- [ ] completed covariance function;
- [ ] completed eigenvalue/eigenvector function;
- [ ] explained-variance calculation;
- [ ] manual PCA interpretation;
- [ ] completed projection function;
- [ ] your own student-ID-derived dataset;
- [ ] correlation analysis;
- [ ] scaling interpretation;
- [ ] full PCA fit;
- [ ] scree and cumulative-variance interpretation;
- [ ] loading interpretation for PC1 and PC2;
- [ ] 2D PCA visualization;
- [ ] reconstruction-error experiment;
- [ ] PCA + K-Means comparison;
- [ ] personalized variance-target challenge;
- [ ] corrected projection debugging task;
- [ ] final synthesis statement;
- [ ] reflection answers;
- [ ] all required code cells executed successfully.

# Assessment — 10 Marks

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality and submission completeness | **1** |
| **Total** | **10** |

### Marking emphasis

Full marks require showing that you understand:

$$
\boxed{
\text{center/scale}
\rightarrow
\text{covariance}
\rightarrow
\text{principal directions}
\rightarrow
\text{projection}
\rightarrow
\text{explained variance}
\rightarrow
\text{useful representation}
}
$$

# Lab 12 Summary

You should now be able to explain and apply Principal Component Analysis.

### PCA representation

$$
Z=X_cV
$$

where the columns of $V$ are principal directions.

### Explained variance

$$
EVR_j
=
\frac{\lambda_j}
{\sum_k \lambda_k}.
$$

### Main lessons

- PCA is unsupervised feature extraction.
- Centering is essential; scaling is often important.
- $PC_1$ captures the direction of maximum variance.
- Later PCs capture remaining orthogonal variation.
- Explained variance measures how much data variation is retained.
- Loadings help interpret how original features contribute to each component.
- Reducing dimensions always trades compression for information loss.
- PCA can help visualization and clustering, but it does not guarantee better clustering.
- High variance is not the same as high supervised predictive value.
- When PCA is part of a supervised pipeline, fit it on training data only.

This completes the **12 core Machine Learning practical labs**.